# LeetCode 1574 — Shortest Subarray to be Removed to Make Array Sorted bằng Python

Link đề: https://leetcode.com/problems/shortest-subarray-to-be-removed-to-make-array-sorted/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Stack, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def findLengthOfShortestSubarray(self, arr: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu rằng sau khi xoá **một đoạn con liên tiếp**, phần còn lại = **tiền tố** + **hậu tố**, và nó không giảm ⇔ tiền tố không giảm, hậu tố không giảm và "chỗ nối" không bị tụt.
2. Viết baseline duyệt mọi đoạn bị xoá `[l, r]` (bản `O(n³)` và bản `O(n²)` dùng bảng tiền tố/hậu tố hợp lệ).
3. Tìm **tiền tố không giảm dài nhất** `arr[0..i]` và **hậu tố không giảm dài nhất** `arr[j..n-1]`, rồi ghép chúng bằng **tìm kiếm nhị phân**.
4. Tối ưu ghép bằng **hai con trỏ** đạt `O(n)` và giải thích vì sao con trỏ phải chỉ đi một chiều.
5. Tránh các bẫy: dùng `<` thay vì `<=`, quên phương án "xoá hết tiền tố/hậu tố", quên trường hợp mảng đã sắp, nhầm `bisect_left`/`bisect_right`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr`. Hãy **xoá một đoạn con liên tiếp** (có thể **rỗng**) sao cho các phần tử còn lại tạo thành dãy **không giảm**.

Trả về **độ dài nhỏ nhất** của đoạn cần xoá.

**Ví dụ 1:** `arr = [1,2,3,10,4,2,3,5]` → `3`. Xoá `[10,4,2]` còn `[1,2,3,3,5]`. Cách khác: xoá `[3,10,4]`.

**Ví dụ 2:** `arr = [5,4,3,2,1]` → `4`. Mảng giảm ngặt nên chỉ giữ được 1 phần tử.

**Ví dụ 3:** `arr = [1,2,3]` → `0`. Đã không giảm sẵn.

**Ràng buộc:**
- `1 ≤ arr.length ≤ 10⁵`
- `0 ≤ arr[i] ≤ 10⁹`

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from typing import List

examples = [([1, 2, 3, 10, 4, 2, 3, 5], 3), ([5, 4, 3, 2, 1], 4), ([1, 2, 3], 0)]
for arr, want in examples:
    print(f"arr = {arr} -> mong đợi {want}")

arr = [1, 2, 3, 10, 4, 2, 3, 5] -> mong đợi 3
arr = [5, 4, 3, 2, 1] -> mong đợi 4
arr = [1, 2, 3] -> mong đợi 0


## 3. Kiến thức nền

### 3.1 "Không giảm" (non-decreasing) và cách kiểm tra

Dãy `a` **không giảm** nếu `a[k] <= a[k+1]` với mọi `k`. Chú ý: cho phép **bằng nhau** (`[1,2,2,3]` vẫn hợp lệ). Kiểm tra bằng `all(...)` và `zip(a, a[1:])`.

In [2]:
def is_non_decreasing(a):
    return all(x <= y for x, y in zip(a, a[1:]))

for a in [[1, 2, 2, 3], [1, 3, 2], [], [7]]:
    print(a, "->", is_non_decreasing(a))

[1, 2, 2, 3] -> True
[1, 3, 2] -> False
[] -> True
[7] -> True


### 3.2 Xoá đoạn con bằng slicing

Xoá đoạn `arr[l..r]` (bao gồm hai đầu) ⇒ còn `arr[:l] + arr[r+1:]`. Độ dài đoạn bị xoá là `r - l + 1`. Đoạn rỗng ứng với không xoá gì.

In [3]:
arr = [1, 2, 3, 10, 4, 2, 3, 5]
l, r = 3, 5
rest = arr[:l] + arr[r + 1:]
print("xoá", arr[l:r + 1], "-> còn", rest, "| không giảm?", is_non_decreasing(rest), "| độ dài xoá:", r - l + 1)

xoá [10, 4, 2] -> còn [1, 2, 3, 3, 5] | không giảm? True | độ dài xoá: 3


### 3.3 Tiền tố / hậu tố không giảm dài nhất

- `i` = chỉ số lớn nhất sao cho `arr[0..i]` không giảm: tăng `i` khi `arr[i] <= arr[i+1]`.
- `j` = chỉ số nhỏ nhất sao cho `arr[j..n-1]` không giảm: giảm `j` khi `arr[j-1] <= arr[j]`.

In [4]:
def longest_prefix_suffix(arr):
    n = len(arr)
    i = 0
    while i + 1 < n and arr[i] <= arr[i + 1]:
        i += 1
    j = n - 1
    while j - 1 >= 0 and arr[j - 1] <= arr[j]:
        j -= 1
    return i, j

for arr, _ in examples:
    i, j = longest_prefix_suffix(arr)
    print(arr, "-> i =", i, arr[:i + 1], "| j =", j, arr[j:])

[1, 2, 3, 10, 4, 2, 3, 5] -> i = 3 [1, 2, 3, 10] | j = 5 [2, 3, 5]
[5, 4, 3, 2, 1] -> i = 0 [5] | j = 4 [1]
[1, 2, 3] -> i = 2 [1, 2, 3] | j = 0 [1, 2, 3]


### 3.4 `bisect_left` vs `bisect_right` trên mảng có phần tử bằng nhau

Trên mảng đã sắp `a`:
- `bisect_left(a, x)` = vị trí đầu tiên có `a[pos] >= x`.
- `bisect_right(a, x)` = vị trí đầu tiên có `a[pos] > x`.

Tham số `lo` giới hạn vùng tìm từ chỉ số `lo` trở đi: `bisect_left(a, x, lo)`. Ta sẽ tìm trong hậu tố `arr[j:]` mà không cần cắt mảng.

In [5]:
a = [0, 2, 3, 3, 3, 7]
for x in [3, 4, 8]:
    print(f"x={x}: bisect_left={bisect_left(a, x)}, bisect_right={bisect_right(a, x)}")
arr = [9, 9, 1, 2, 3]
print("tìm từ lo=2 giá trị >= 2:", bisect_left(arr, 2, 2))

x=3: bisect_left=2, bisect_right=5
x=4: bisect_left=5, bisect_right=5
x=8: bisect_left=6, bisect_right=6
tìm từ lo=2 giá trị >= 2: 3


### 3.5 Kỹ thuật hai con trỏ cùng chiều

Hai chỉ số `l`, `r` cùng **chỉ tiến**, mỗi con trỏ đi tối đa `n` bước ⇒ tổng `O(n)`. Điều kiện để dùng: khi `l` tăng thì vị trí `r` tốt nhất **không giảm** (tính đơn điệu).

In [6]:
# Ví dụ nhỏ: với mỗi phần tử của a (tăng dần), tìm phần tử đầu tiên >= nó trong b (tăng dần)
a, b = [1, 4, 4, 9], [2, 3, 5, 8, 10]
r = 0
for x in a:
    while r < len(b) and b[r] < x:
        r += 1
    print(f"x={x}: phần tử đầu tiên >= x trong b ở vị trí {r} ->", b[r] if r < len(b) else None)

x=1: phần tử đầu tiên >= x trong b ở vị trí 0 -> 2
x=4: phần tử đầu tiên >= x trong b ở vị trí 2 -> 5
x=4: phần tử đầu tiên >= x trong b ở vị trí 2 -> 5
x=9: phần tử đầu tiên >= x trong b ở vị trí 4 -> 10


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `arr = [1, 2, 3, 10, 4, 2, 3, 5]` (`n = 8`).

- Tiền tố không giảm dài nhất: `[1, 2, 3, 10]` ⇒ `i = 3`.
- Hậu tố không giảm dài nhất: `[2, 3, 5]` ⇒ `j = 5`.

Đoạn bị xoá phải "bao" phần lộn xộn ở giữa (`arr[i+1..j-1] = [4]` và hơn nữa). Phần còn lại là một **tiền tố của `[1,2,3,10]`** nối với **hậu tố của `[2,3,5]`**. Các lựa chọn:

| giữ tiền tố | giữ hậu tố (bắt đầu ≥ phần tử cuối tiền tố) | xoá |
|---|---|---|
| `[]` | `[2,3,5]` (từ j=5) | 5 |
| `[1,2,3,10]` | `[]` | 4 |
| `[1]` | `[2,3,5]` | 4 |
| `[1,2]` | `[2,3,5]` | 3 |
| `[1,2,3]` | `[3,5]` | 3 |
| `[1,2,3,10]` | `[]` (không có phần tử ≥ 10) | 4 |

Nhỏ nhất là **3**. Ý tưởng: với mỗi điểm kết thúc tiền tố `l`, tìm điểm bắt đầu hậu tố `r ≥ j` **sớm nhất** có `arr[r] >= arr[l]`.

In [7]:
arr = [1, 2, 3, 10, 4, 2, 3, 5]
n = len(arr)
i, j = longest_prefix_suffix(arr)
print("i =", i, "| j =", j)
print("xoá hết sau tiền tố:", n - 1 - i, "| xoá hết trước hậu tố:", j)
for l in range(i + 1):
    r = j
    while r < n and arr[r] < arr[l]:
        r += 1
    print(f"giữ {arr[:l + 1]} + {arr[r:]} -> xoá {r - l - 1}")

i = 3 | j = 5
xoá hết sau tiền tố: 4 | xoá hết trước hậu tố: 5
giữ [1] + [2, 3, 5] -> xoá 4
giữ [1, 2] + [2, 3, 5] -> xoá 3
giữ [1, 2, 3] + [3, 5] -> xoá 3
giữ [1, 2, 3, 10] + [] -> xoá 4


## 5. Cách 1 — Brute Force / baseline

### 5.1 Bản ngây thơ `O(n³)`

Thử mọi độ dài `L = 0, 1, 2, …` và mọi vị trí bắt đầu; đoạn đầu tiên làm phần còn lại không giảm cho ta đáp án.

In [8]:
def brute_force_naive(arr: List[int]) -> int:
    n = len(arr)
    for L in range(n):                     # thử độ dài xoá tăng dần
        for l in range(n - L + 1):
            rest = arr[:l] + arr[l + L:]
            if is_non_decreasing(rest):
                return L
    return n - 1                           # không tới được vì L = n-1 luôn hợp lệ

for arr, want in examples:
    print(arr, "->", brute_force_naive(arr), "| mong đợi", want)

[1, 2, 3, 10, 4, 2, 3, 5] -> 3 | mong đợi 3
[5, 4, 3, 2, 1] -> 4 | mong đợi 4
[1, 2, 3] -> 0 | mong đợi 0


### 5.2 Bản `O(n²)` với bảng tiền tố/hậu tố hợp lệ

Tiền tính `pre_ok[l]` = `arr[:l]` không giảm, `suf_ok[r]` = `arr[r:]` không giảm. Xoá `arr[l..r-1]` (giữ `arr[:l]` và `arr[r:]`, `0 ≤ l ≤ r ≤ n`) hợp lệ ⇔ `pre_ok[l]` và `suf_ok[r]` và (`l == 0` hoặc `r == n` hoặc `arr[l-1] <= arr[r]`).

In [9]:
def brute_force(arr: List[int]) -> int:
    n = len(arr)
    pre_ok = [True] * (n + 1)
    for l in range(2, n + 1):
        pre_ok[l] = pre_ok[l - 1] and arr[l - 2] <= arr[l - 1]
    suf_ok = [True] * (n + 1)
    for r in range(n - 2, -1, -1):
        suf_ok[r] = suf_ok[r + 1] and arr[r] <= arr[r + 1]
    best = n
    for l in range(n + 1):
        if not pre_ok[l]:
            break
        for r in range(l, n + 1):
            if suf_ok[r] and (l == 0 or r == n or arr[l - 1] <= arr[r]):
                best = min(best, r - l)
                break                      # r nhỏ nhất cho l này đã tìm thấy
    return best

for arr, want in examples:
    print(arr, "->", brute_force(arr), "| mong đợi", want)

[1, 2, 3, 10, 4, 2, 3, 5] -> 3 | mong đợi 3
[5, 4, 3, 2, 1] -> 4 | mong đợi 4
[1, 2, 3] -> 0 | mong đợi 0


**Độ phức tạp:** bản 5.1 là `O(n³)`; bản 5.2 là `O(n²)` thời gian, `O(n)` bộ nhớ.

**Nút thắt:** với `n = 10⁵`, `O(n²) = 10¹⁰` — quá chậm. Quan sát: `pre_ok[l]` chỉ đúng với `l ≤ i+1`, `suf_ok[r]` chỉ đúng với `r ≥ j`. Ta chỉ cần xét `l` trong tiền tố, `r` trong hậu tố, và với mỗi `l` tìm `r` **nhanh**.

In [10]:
rng = random.Random(1)
arr_big = sorted(rng.randint(0, 50) for _ in range(1500)) + [0] + sorted(rng.randint(0, 50) for _ in range(1500))
t0 = time.perf_counter()
res = brute_force(arr_big)
print(f"n={len(arr_big)}: {res}, mất {time.perf_counter() - t0:.3f}s")

n=3001: 1460, mất 0.155s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** phần còn lại là `arr[0..l] + arr[r..n-1]` (với `l < r`, hoặc một trong hai rỗng). Nó không giảm ⇔
1. `arr[0..l]` không giảm ⇔ `l ≤ i`;
2. `arr[r..n-1]` không giảm ⇔ `r ≥ j`;
3. chỗ nối: `arr[l] <= arr[r]`.

**Insight 2 — ba nhóm phương án:**
- giữ nguyên tiền tố, xoá hết phần sau: xoá `n - 1 - i`;
- giữ nguyên hậu tố, xoá hết phần trước: xoá `j`;
- giữ `arr[0..l]` và `arr[r..]` với `l ≤ i`, `r ≥ j`, `arr[l] <= arr[r]`: xoá `r - l - 1`.

**Insight 3 — đơn điệu:** hậu tố `arr[j..]` đã sắp ⇒ với mỗi `l`, `r` tốt nhất là **vị trí đầu tiên ≥ j có `arr[r] >= arr[l]`** (tìm nhị phân). Thêm nữa, khi `l` tăng thì `arr[l]` không giảm ⇒ `r` tốt nhất **không lùi** ⇒ hai con trỏ.

**Quy tắc nhanh:** *nếu* `i == n - 1` *thì* trả 0 ngay; *nếu* `arr[r] < arr[l]` *thì* tăng `r`; *nếu* `r == n` *thì* mọi `l` lớn hơn cũng không ghép được ⇒ dừng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền tố/hậu tố + tìm kiếm nhị phân

**Đoạn 1:** tìm `i`; nếu `i == n - 1` trả `0`.

**Đoạn 2:** tìm `j`; khởi tạo `ans = min(n - 1 - i, j)`.

**Đoạn 3:** với mỗi `l = 0..i`, `r = bisect_left(arr, arr[l], lo=j)` — vị trí đầu tiên trong `arr[j:]` có giá trị `>= arr[l]` (hậu tố đã sắp nên tìm nhị phân hợp lệ). Cập nhật `ans = min(ans, r - l - 1)`.

Lưu ý: `bisect_left(arr, x, lo=j)` chỉ yêu cầu `arr[j:]` sắp xếp — phần trước `j` không được xem tới.

In [11]:
def shortest_binary_search(arr: List[int]) -> int:
    n = len(arr)
    i = 0
    while i + 1 < n and arr[i] <= arr[i + 1]:
        i += 1
    if i == n - 1:
        return 0
    j = n - 1
    while arr[j - 1] <= arr[j]:            # j > i nên j-1 >= 0
        j -= 1
    ans = min(n - 1 - i, j)
    for l in range(i + 1):
        r = bisect_left(arr, arr[l], j)    # đầu tiên trong arr[j:] có giá trị >= arr[l]
        ans = min(ans, r - l - 1)
    return ans

for arr, want in examples:
    print(arr, "->", shortest_binary_search(arr), "| mong đợi", want)

[1, 2, 3, 10, 4, 2, 3, 5] -> 3 | mong đợi 3
[5, 4, 3, 2, 1] -> 4 | mong đợi 4
[1, 2, 3] -> 0 | mong đợi 0


Code nộp LeetCode cho cách 2:

```python
class Solution:
    def findLengthOfShortestSubarray(self, arr: List[int]) -> int:
        n = len(arr)
        i = 0
        while i + 1 < n and arr[i] <= arr[i + 1]:
            i += 1
        if i == n - 1:
            return 0
        j = n - 1
        while arr[j - 1] <= arr[j]:
            j -= 1
        ans = min(n - 1 - i, j)
        for l in range(i + 1):
            r = bisect_left(arr, arr[l], j)
            ans = min(ans, r - l - 1)
        return ans
```

**Độ phức tạp:** `O(n log n)` thời gian (mỗi `l` một lần tìm nhị phân), `O(1)` bộ nhớ phụ.

Vì sao vòng `while arr[j - 1] <= arr[j]` không cần kiểm tra `j - 1 >= 0`? Vì mảng **chưa** sắp (`i < n-1`) nên tồn tại `arr[i] > arr[i+1]`; vòng lặp chắc chắn dừng khi `j = i + 1` hoặc sớm hơn.

### 7.2 Cách 3 — Hai con trỏ `O(n)`

Thay tìm nhị phân bằng con trỏ `r` chạy cùng chiều với `l`:

```
r = j
for l in 0..i:
    while r < n and arr[r] < arr[l]: r += 1
    if r == n: break          # các l sau còn lớn hơn -> cũng không ghép được
    ans = min(ans, r - l - 1)
```

Vì `arr[0..i]` không giảm, `arr[l]` tăng dần theo `l` ⇒ ngưỡng `arr[r] >= arr[l]` ngày càng chặt ⇒ `r` chỉ cần tiến, không bao giờ lùi. Mỗi con trỏ đi tối đa `n` bước.

In [12]:
def shortest_two_pointers(arr: List[int]) -> int:
    n = len(arr)
    i = 0
    while i + 1 < n and arr[i] <= arr[i + 1]:
        i += 1
    if i == n - 1:
        return 0
    j = n - 1
    while arr[j - 1] <= arr[j]:
        j -= 1
    ans = min(n - 1 - i, j)
    r = j
    for l in range(i + 1):
        while r < n and arr[r] < arr[l]:
            r += 1
        if r == n:
            break
        ans = min(ans, r - l - 1)
    return ans

for arr, want in examples:
    print(arr, "->", shortest_two_pointers(arr), "| mong đợi", want)

[1, 2, 3, 10, 4, 2, 3, 5] -> 3 | mong đợi 3
[5, 4, 3, 2, 1] -> 4 | mong đợi 4
[1, 2, 3] -> 0 | mong đợi 0


Code nộp LeetCode cho cách 3 (giống phần 9):

```python
class Solution:
    def findLengthOfShortestSubarray(self, arr: List[int]) -> int:
        n = len(arr)
        i = 0
        while i + 1 < n and arr[i] <= arr[i + 1]:
            i += 1
        if i == n - 1:
            return 0
        j = n - 1
        while arr[j - 1] <= arr[j]:
            j -= 1
        ans = min(n - 1 - i, j)
        r = j
        for l in range(i + 1):
            while r < n and arr[r] < arr[l]:
                r += 1
            if r == n:
                break
            ans = min(ans, r - l - 1)
        return ans
```

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ.

### 7.3 Cách 3' — Hai con trỏ duyệt theo hậu tố (góc nhìn đối xứng)

Cũng có thể cố định điểm bắt đầu hậu tố `r` (từ `j` đến `n-1`) và tìm `l` **lớn nhất** trong tiền tố có `arr[l] <= arr[r]`. Khi `r` tăng, `arr[r]` không giảm ⇒ `l` chỉ tiến. Đây là cùng một ý tưởng, hữu ích khi muốn viết theo kiểu "với mỗi phần tử hậu tố".

In [13]:
def shortest_two_pointers_suffix(arr: List[int]) -> int:
    n = len(arr)
    i, j = longest_prefix_suffix(arr)
    if i == n - 1:
        return 0
    ans = min(n - 1 - i, j)
    l = -1                                  # l = chỉ số cuối của tiền tố được giữ (-1 = không giữ gì)
    for r in range(j, n):
        while l + 1 <= i and arr[l + 1] <= arr[r]:
            l += 1
        ans = min(ans, r - l - 1)
    return ans

for arr, want in examples:
    print(arr, "->", shortest_two_pointers_suffix(arr), "| mong đợi", want)

[1, 2, 3, 10, 4, 2, 3, 5] -> 3 | mong đợi 3
[5, 4, 3, 2, 1] -> 4 | mong đợi 4
[1, 2, 3] -> 0 | mong đợi 0


## 8. TRACE từng bước

Ví dụ 1 với cách 3: `arr = [1, 2, 3, 10, 4, 2, 3, 5]`, `n = 8`, `i = 3`, `j = 5`, `ans = min(8-1-3, 5) = 4`.

| l | arr[l] | r trước | r sau (arr[r] ≥ arr[l]) | arr[r] | xoá = r-l-1 | ans |
|---|---|---|---|---|---|---|
| 0 | 1 | 5 | 5 | 2 | 4 | 4 |
| 1 | 2 | 5 | 5 | 2 | 3 | 3 |
| 2 | 3 | 5 | 6 | 3 | 3 | 3 |
| 3 | 10 | 6 | 8 (= n) | — | dừng | **3** |

In [14]:
def trace_two_pointers(arr):
    n = len(arr)
    i, j = longest_prefix_suffix(arr)
    print(f"arr={arr}, n={n}, i={i} (tiền tố {arr[:i + 1]}), j={j} (hậu tố {arr[j:]})")
    if i == n - 1:
        print("đã không giảm -> 0")
        return 0
    ans = min(n - 1 - i, j)
    print(f"khởi tạo ans = min(n-1-i={n - 1 - i}, j={j}) = {ans}")
    r = j
    for l in range(i + 1):
        r0 = r
        while r < n and arr[r] < arr[l]:
            r += 1
        if r == n:
            print(f"  l={l} arr[l]={arr[l]}: r {r0}->{r} = n -> dừng")
            break
        ans = min(ans, r - l - 1)
        print(f"  l={l} arr[l]={arr[l]}: r {r0}->{r} (arr[r]={arr[r]}), xoá arr[{l + 1}..{r - 1}]={arr[l + 1:r]} dài {r - l - 1}, ans={ans}")
    print("kết quả:", ans)
    return ans

trace_two_pointers([1, 2, 3, 10, 4, 2, 3, 5])

arr=[1, 2, 3, 10, 4, 2, 3, 5], n=8, i=3 (tiền tố [1, 2, 3, 10]), j=5 (hậu tố [2, 3, 5])
khởi tạo ans = min(n-1-i=4, j=5) = 4
  l=0 arr[l]=1: r 5->5 (arr[r]=2), xoá arr[1..4]=[2, 3, 10, 4] dài 4, ans=4
  l=1 arr[l]=2: r 5->5 (arr[r]=2), xoá arr[2..4]=[3, 10, 4] dài 3, ans=3
  l=2 arr[l]=3: r 5->6 (arr[r]=3), xoá arr[3..5]=[10, 4, 2] dài 3, ans=3
  l=3 arr[l]=10: r 6->8 = n -> dừng
kết quả: 3


3

In [15]:
trace_two_pointers([5, 4, 3, 2, 1])
print()
trace_two_pointers([1, 3, 5, 2, 2, 4, 6, 0, 1, 7])

arr=[5, 4, 3, 2, 1], n=5, i=0 (tiền tố [5]), j=4 (hậu tố [1])
khởi tạo ans = min(n-1-i=4, j=4) = 4
  l=0 arr[l]=5: r 4->5 = n -> dừng
kết quả: 4

arr=[1, 3, 5, 2, 2, 4, 6, 0, 1, 7], n=10, i=2 (tiền tố [1, 3, 5]), j=7 (hậu tố [0, 1, 7])
khởi tạo ans = min(n-1-i=7, j=7) = 7
  l=0 arr[l]=1: r 7->8 (arr[r]=1), xoá arr[1..7]=[3, 5, 2, 2, 4, 6, 0] dài 7, ans=7
  l=1 arr[l]=3: r 8->9 (arr[r]=7), xoá arr[2..8]=[5, 2, 2, 4, 6, 0, 1] dài 7, ans=7
  l=2 arr[l]=5: r 9->9 (arr[r]=7), xoá arr[3..8]=[2, 2, 4, 6, 0, 1] dài 6, ans=6
kết quả: 6


6

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def findLengthOfShortestSubarray(self, arr: List[int]) -> int:
        n = len(arr)
        i = 0
        while i + 1 < n and arr[i] <= arr[i + 1]:
            i += 1
        if i == n - 1:
            return 0
        j = n - 1
        while arr[j - 1] <= arr[j]:
            j -= 1
        ans = min(n - 1 - i, j)
        r = j
        for l in range(i + 1):
            while r < n and arr[r] < arr[l]:
                r += 1
            if r == n:
                break
            ans = min(ans, r - l - 1)
        return ans

sol = Solution()
for arr, want in examples:
    print(arr, "->", sol.findLengthOfShortestSubarray(arr), "| mong đợi", want)

[1, 2, 3, 10, 4, 2, 3, 5] -> 3 | mong đợi 3
[5, 4, 3, 2, 1] -> 4 | mong đợi 4
[1, 2, 3] -> 0 | mong đợi 0


## 10. Vì sao thuật toán đúng

**Đủ phương án.** Mọi cách xoá đoạn `arr[a..b]` để lại `arr[0..a-1]` và `arr[b+1..n-1]`.
- Nếu `a = 0`: phần còn lại là hậu tố `arr[b+1..]`, không giảm ⇔ `b + 1 ≥ j` ⇒ tốt nhất xoá `j` phần tử.
- Nếu `b = n-1`: tương tự, tốt nhất xoá `n-1-i`.
- Nếu không: đặt `l = a-1`, `r = b+1`; cần `l ≤ i`, `r ≥ j`, `arr[l] ≤ arr[r]`. Với `l` cố định, xoá `r - l - 1` nhỏ nhất ⇔ `r` nhỏ nhất thoả điều kiện ⇔ `r` = vị trí đầu tiên `≥ j` có `arr[r] ≥ arr[l]` (vì `arr[j..]` không giảm, tập `r` thoả điều kiện là một đoạn kéo tới cuối). Lưu ý `r > l` tự động vì `r ≥ j > i ≥ l`.

**Hai con trỏ không bỏ sót.** Gọi `R(l)` là `r` tốt nhất cho `l`. Vì `arr[l] ≤ arr[l+1]` (trong tiền tố) nên mọi `r` thoả cho `l+1` cũng thoả cho `l` ⇒ `R(l) ≤ R(l+1)`. Do đó con trỏ `r` khi xử lý `l+1` có thể bắt đầu từ `R(l)` — các vị trí trước đó chắc chắn không thoả. Khi `r == n` với `l` nào đó thì `R(l') = n` với mọi `l' > l` ⇒ dừng an toàn.

**Trường hợp đã sắp:** `i = n-1` ⇒ không cần xoá ⇒ 0.

## 11. Phân tích Time / Space

Gọi `n = len(arr)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force ngây thơ | `O(n³)` | `O(n)` (mảng còn lại) |
| 1'. Brute force + bảng hợp lệ | `O(n²)` | `O(n)` |
| 2. Tiền tố/hậu tố + nhị phân | `O(n log n)` | `O(1)` |
| 3. Hai con trỏ | `O(n)` | `O(1)` |

Cách 3: tìm `i` và `j` mất `O(n)`; trong vòng ghép, `l` tăng tối đa `i+1` lần và `r` tăng tối đa `n - j` lần ⇒ tổng `O(n)`. Không có khác biệt average/worst.

## 12. Bộ test

In [17]:
ALL = [brute_force, shortest_binary_search, shortest_two_pointers, shortest_two_pointers_suffix,
       Solution().findLengthOfShortestSubarray]

for f in ALL:
    for arr, want in examples:
        assert f(list(arr)) == want, (f.__name__, arr)
print("Ví dụ đề: OK")

edge = [
    ([7], 0),                        # n = 1
    ([2, 1], 1),                     # n = 2 giảm
    ([1, 1, 1, 1], 0),               # toàn bằng nhau (không giảm)
    ([10, 1, 2, 3], 1),              # xoá phần tử đầu là tốt nhất
    ([1, 2, 3, 0], 1),               # xoá phần tử cuối là tốt nhất
    ([1, 2, 3, 3, 0, 3, 4], 1),      # có phần tử bằng nhau ở chỗ nối
    ([2, 2, 2, 1, 1, 1], 3),
    ([6, 3, 10, 11, 15, 20, 13, 3, 18, 12], 8),
]
for f in ALL:
    for arr, want in edge:
        assert f(list(arr)) == want, (f.__name__, arr, f(list(arr)), want)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [18]:
# Ngẫu nhiên seed cố định, đối chiếu với brute force ngây thơ O(n^3)
rng = random.Random(1574)
for _ in range(2000):
    n = rng.randint(1, 12)
    hi = rng.choice([1, 3, 10, 10**9])
    arr = [rng.randint(0, hi) for _ in range(n)]
    want = brute_force_naive(arr)
    for f in ALL:
        assert f(list(arr)) == want, (f.__name__, arr)
print("2000 mảng ngẫu nhiên: OK")

2000 mảng ngẫu nhiên: OK


In [19]:
# Mảng "gần sắp": ghép hai dãy đã sắp + nhiễu ở giữa (dạng dễ lộ lỗi chỗ nối)
rng = random.Random(7)
for _ in range(1000):
    a = sorted(rng.randint(0, 8) for _ in range(rng.randint(0, 6)))
    mid = [rng.randint(0, 8) for _ in range(rng.randint(0, 3))]
    b = sorted(rng.randint(0, 8) for _ in range(rng.randint(0, 6)))
    arr = a + mid + b
    if not arr:
        continue
    want = brute_force_naive(arr)
    for f in ALL:
        assert f(list(arr)) == want, (f.__name__, arr)
print("1000 mảng gần sắp: OK")

1000 mảng gần sắp: OK


In [20]:
# Test lớn đúng ràng buộc n = 10^5
rng = random.Random(99)
big_cases = [
    list(range(10**5)),                                       # đã sắp -> 0
    list(range(10**5, 0, -1)),                                 # giảm ngặt -> n-1
    sorted(rng.randint(0, 10**9) for _ in range(50000)) + sorted(rng.randint(0, 10**9) for _ in range(50000)),
    [rng.randint(0, 10**9) for _ in range(10**5)],
]
for arr in big_cases:
    t0 = time.perf_counter()
    r2 = shortest_binary_search(arr)
    r3 = shortest_two_pointers(arr)
    assert r2 == r3
    print(f"n={len(arr)}: {r3}  ({time.perf_counter() - t0:.3f}s cho 2 cách)")
assert shortest_two_pointers(big_cases[0]) == 0 and shortest_two_pointers(big_cases[1]) == 10**5 - 1

n=100000: 0  (0.015s cho 2 cách)
n=100000: 99999  (0.000s cho 2 cách)


n=100000: 49952  (0.043s cho 2 cách)
n=100000: 99997  (0.000s cho 2 cách)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `<` (tăng ngặt) thay vì `<=` (không giảm)

Đề yêu cầu **không giảm**, nên các phần tử bằng nhau được phép đứng cạnh nhau.

In [21]:
def wrong_strict(arr):
    n = len(arr)
    i = 0
    while i + 1 < n and arr[i] < arr[i + 1]:      # SAI: phải là <=
        i += 1
    if i == n - 1:
        return 0
    j = n - 1
    while j > 0 and arr[j - 1] < arr[j]:          # SAI
        j -= 1
    ans = min(n - 1 - i, j)
    r = j
    for l in range(i + 1):
        while r < n and arr[r] <= arr[l]:          # SAI theo
            r += 1
        if r == n:
            break
        ans = min(ans, r - l - 1)
    return ans

for arr in [[1, 2, 2, 3], [1, 1, 1]]:
    print(arr, "-> sai:", wrong_strict(arr), "| đúng:", Solution().findLengthOfShortestSubarray(arr))

[1, 2, 2, 3] -> sai: 1 | đúng: 0
[1, 1, 1] -> sai: 2 | đúng: 0


### Lỗi 2 — Quên hai phương án "xoá hết phần sau tiền tố" và "xoá hết phần trước hậu tố"

Nếu khởi tạo `ans = n` và chỉ xét ghép tiền tố–hậu tố, ta bỏ lỡ trường hợp **không giữ gì** ở một phía.

In [22]:
def wrong_no_init(arr):
    n = len(arr)
    i, j = longest_prefix_suffix(arr)
    if i == n - 1:
        return 0
    ans = n                                       # SAI: thiếu min(n-1-i, j)
    r = j
    for l in range(i + 1):
        while r < n and arr[r] < arr[l]:
            r += 1
        if r == n:
            break
        ans = min(ans, r - l - 1)
    return ans

for arr in [[10, 1, 2, 3], [1, 2, 3, 0]]:
    print(arr, "-> sai:", wrong_no_init(arr), "| đúng:", Solution().findLengthOfShortestSubarray(arr))

[10, 1, 2, 3] -> sai: 4 | đúng: 1
[1, 2, 3, 0] -> sai: 4 | đúng: 1


### Lỗi 3 — Không return sớm khi mảng đã sắp

Khi `i == n-1`, tính `j` cho ra `0`, con trỏ `r` bắt đầu ở `0 ≤ l`, và `r - l - 1` có thể **âm**.

In [23]:
def wrong_no_early_return(arr):
    n = len(arr)
    i, j = longest_prefix_suffix(arr)
    ans = min(n - 1 - i, j)                       # thiếu: if i == n-1: return 0 (vẫn = 0 ở đây...)
    r = j
    for l in range(i + 1):
        while r < n and arr[r] < arr[l]:
            r += 1
        if r == n:
            break
        ans = min(ans, r - l - 1)                 # ... nhưng r có thể = l -> -1
    return ans

for arr in [[1, 2, 3], [5]]:
    print(arr, "-> sai:", wrong_no_early_return(arr), "| đúng:", Solution().findLengthOfShortestSubarray(arr))

[1, 2, 3] -> sai: -1 | đúng: 0
[5] -> sai: -1 | đúng: 0


### Lỗi 4 — Dùng `bisect_right` thay vì `bisect_left`

Ta cần `arr[r] >= arr[l]` (cho phép bằng). `bisect_right` tìm `arr[r] > arr[l]` ⇒ bỏ lỡ chỗ nối bằng nhau, xoá thừa.

In [24]:
def wrong_bisect_right(arr):
    n = len(arr)
    i, j = longest_prefix_suffix(arr)
    if i == n - 1:
        return 0
    ans = min(n - 1 - i, j)
    for l in range(i + 1):
        r = bisect_right(arr, arr[l], j)          # SAI: phải là bisect_left
        ans = min(ans, r - l - 1)
    return ans

arr = [1, 2, 3, 3, 0, 3, 4]
print(arr, "-> sai:", wrong_bisect_right(arr), "| đúng:", Solution().findLengthOfShortestSubarray(arr))

[1, 2, 3, 3, 0, 3, 4] -> sai: 2 | đúng: 1


### Lỗi 5 — Nghĩ rằng chỉ cần xoá "phần lộn xộn ở giữa" `arr[i+1..j-1]`

Phần giữa có thể nhỏ, nhưng chỗ nối `arr[i] <= arr[j]` chưa chắc đúng; thường phải xoá thêm.

In [25]:
def wrong_middle_only(arr):
    i, j = longest_prefix_suffix(arr)
    return max(0, j - i - 1)                      # SAI: bỏ qua điều kiện chỗ nối

for arr in [[1, 2, 3, 10, 4, 2, 3, 5], [5, 4, 3, 2, 1]]:
    print(arr, "-> sai:", wrong_middle_only(arr), "| đúng:", Solution().findLengthOfShortestSubarray(arr))

[1, 2, 3, 10, 4, 2, 3, 5] -> sai: 1 | đúng: 3
[5, 4, 3, 2, 1] -> sai: 3 | đúng: 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xoá đúng một đoạn liên tiếp"** ⇒ phần còn lại = **tiền tố + hậu tố**. Phân tích riêng từng phía rồi xử lý chỗ nối.
- Điều kiện "phần còn lại có tính chất P" thường tách được thành: P(tiền tố) ∧ P(hậu tố) ∧ điều kiện nối.
- Hai phía đã sắp ⇒ chỗ nối tìm bằng **nhị phân** hoặc **hai con trỏ**.
- Đừng quên các phương án biên: không giữ gì ở bên trái / bên phải, và không xoá gì.
- Kiểm tra nhanh: mảng đã sắp → 0; giảm ngặt → `n-1`; `[10,1,2,3]` → 1; `[1,2,3,0]` → 1.

## 15. Biến thể của bài

1. **Dãy tăng ngặt** thay vì không giảm: đổi `<=` thành `<` ở tiền tố/hậu tố và yêu cầu `arr[l] < arr[r]` ở chỗ nối (bài tập 4).
2. **Trả về chính đoạn cần xoá** `(l, r)`, không chỉ độ dài (bài tập 3).
3. **LC 581 — Shortest Unsorted Continuous Subarray:** đoạn ngắn nhất cần **sắp xếp** (không phải xoá) để cả mảng sắp (bài tập 5).
4. **LC 1590 — Make Sum Divisible by P:** xoá đoạn ngắn nhất để tổng còn lại chia hết cho `p` (tiền tố + hash modulo).
5. **LC 2972/2970 — Count the Number of Incremovable Subarrays:** đếm **số** đoạn xoá được để còn tăng ngặt — dùng đúng khung tiền tố/hậu tố + hai con trỏ, cộng dồn số `r` hợp lệ.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi đoạn xoá (tăng dần độ dài) | `O(n³)` / `O(n²)` | `O(n)` | kiểm thử, `n` nhỏ |
| 2. Nhị phân | tiền tố/hậu tố sắp, với mỗi `l` tìm `r` bằng `bisect_left` | `O(n log n)` | `O(1)` | dễ nghĩ ra, đủ nhanh |
| 3. Hai con trỏ | `r` chỉ tiến khi `l` tăng | `O(n)` | `O(1)` | tối ưu, lời giải chuẩn |
| 3'. Hai con trỏ theo hậu tố | với mỗi `r`, `l` lớn nhất có `arr[l] <= arr[r]` | `O(n)` | `O(1)` | khi muốn đếm theo hậu tố (bài đếm đoạn) |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().findLengthOfShortestSubarray([1, 3, 2, 4])` trả về bao nhiêu?

**Bài 2 (tính tay):** Với `arr = [2, 2, 2, 1, 1, 1]`, hãy tính `i`, `j` và đáp án. Viết hàm `ex2_info(arr)` trả về `(i, j, ans)`.

**Bài 3 (điền TODO):** Viết `ex3_which_subarray(arr)` trả về cặp `(a, b)` là **đoạn cần xoá** (bao gồm hai đầu) có độ dài nhỏ nhất; trả `None` nếu không cần xoá.

**Bài 4 (tự viết lại — biến thể):** Viết `ex4_shortest_strict(arr)`: độ dài đoạn ngắn nhất cần xoá để phần còn lại **tăng ngặt**.

**Bài 5 (mini challenge — LC 581):** Viết `ex5_shortest_unsorted(nums)` trả về độ dài đoạn ngắn nhất cần **sắp xếp lại** (tại chỗ) để toàn bộ mảng không giảm, trong `O(n)`.

In [26]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_which_subarray(arr):
    n = len(arr)
    i, j = longest_prefix_suffix(arr)
    if i == n - 1:
        return None
    best = None   # (độ dài, a, b)
    # TODO: xét phương án xoá arr[i+1 .. n-1] và arr[0 .. j-1]
    # TODO: hai con trỏ: với mỗi l, đoạn xoá là arr[l+1 .. r-1]
    # TODO: trả về (a, b) của phương án ngắn nhất
    return best

In [27]:
# Bài 4 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex4_shortest_strict(arr):
    # TODO: tiền tố tăng ngặt dài nhất (dùng <), hậu tố tăng ngặt dài nhất
    # TODO: chỗ nối cần arr[l] < arr[r]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tìm `i` và `j`. (2) `i = 1`, `j = 2`. (3) Thử giữ `[1]` + `[2,4]`, `[1,3]` + `[4]`.

**Bài 2.** (1) Tiền tố `[2,2,2]` ⇒ `i = 2`. (2) Hậu tố `[1,1,1]` ⇒ `j = 3`. (3) Mọi phần tử tiền tố đều > mọi phần tử hậu tố ⇒ chỉ có thể giữ một phía.

**Bài 3.** (1) Giống cách 3 nhưng lưu thêm vị trí. (2) Xoá `arr[i+1..n-1]` có `(a, b) = (i+1, n-1)`; xoá `arr[0..j-1]` có `(0, j-1)`. (3) Ghép `l`, `r` ⇒ `(l+1, r-1)`; độ dài `r-l-1` phải `> 0` (luôn đúng khi chưa sắp).

**Bài 4.** (1) Đổi mọi `<=` thành `<` trong việc tìm `i`, `j`. (2) Con trỏ `r` tiến khi `arr[r] <= arr[l]`. (3) Kiểm tra bằng brute force với hàm `is_strict`.

**Bài 5.** (1) Duyệt trái→phải giữ `max`; vị trí cuối cùng mà `nums[k] < max` là biên phải. (2) Duyệt phải→trái giữ `min`; vị trí cuối cùng mà `nums[k] > min` là biên trái. (3) Nếu không tìm thấy biên ⇒ 0.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
ans1 = Solution().findLengthOfShortestSubarray([1, 3, 2, 4])
print("Bài 1:", ans1)
assert ans1 == 1 == brute_force_naive([1, 3, 2, 4])

# Bài 2
def ex2_info(arr):
    i, j = longest_prefix_suffix(arr)
    return i, j, Solution().findLengthOfShortestSubarray(arr)

print("Bài 2:", ex2_info([2, 2, 2, 1, 1, 1]))
assert ex2_info([2, 2, 2, 1, 1, 1]) == (2, 3, 3)
print("Bài 1, 2: OK")

Bài 1: 1
Bài 2: (2, 3, 3)
Bài 1, 2: OK


In [29]:
# Bài 3
def ex3_which_subarray_sol(arr):
    n = len(arr)
    i, j = longest_prefix_suffix(arr)
    if i == n - 1:
        return None
    cands = [(n - 1 - i, i + 1, n - 1), (j, 0, j - 1)]
    r = j
    for l in range(i + 1):
        while r < n and arr[r] < arr[l]:
            r += 1
        if r == n:
            break
        cands.append((r - l - 1, l + 1, r - 1))
    _, a, b = min(cands)
    return a, b

print("Bài 3:", ex3_which_subarray_sol([1, 2, 3, 10, 4, 2, 3, 5]), ex3_which_subarray_sol([1, 2, 3]))
rng = random.Random(3)
for _ in range(1500):
    arr = [rng.randint(0, 6) for _ in range(rng.randint(1, 12))]
    res = ex3_which_subarray_sol(arr)
    want = brute_force_naive(arr)
    if res is None:
        assert want == 0 and is_non_decreasing(arr)
    else:
        a, b = res
        assert b - a + 1 == want and is_non_decreasing(arr[:a] + arr[b + 1:]), (arr, res)
print("Bài 3: OK")

Bài 3: (2, 4) None
Bài 3: OK


In [30]:
# Bài 4
def ex4_shortest_strict_sol(arr):
    n = len(arr)
    i = 0
    while i + 1 < n and arr[i] < arr[i + 1]:
        i += 1
    if i == n - 1:
        return 0
    j = n - 1
    while arr[j - 1] < arr[j]:
        j -= 1
    ans = min(n - 1 - i, j)
    r = j
    for l in range(i + 1):
        while r < n and arr[r] <= arr[l]:
            r += 1
        if r == n:
            break
        ans = min(ans, r - l - 1)
    return ans

def is_strict(a):
    return all(x < y for x, y in zip(a, a[1:]))

def ex4_brute(arr):
    n = len(arr)
    for L in range(n):
        for l in range(n - L + 1):
            if is_strict(arr[:l] + arr[l + L:]):
                return L
    return n - 1

rng = random.Random(4)
for _ in range(1500):
    arr = [rng.randint(0, 5) for _ in range(rng.randint(1, 11))]
    assert ex4_shortest_strict_sol(arr) == ex4_brute(arr), arr
print("Bài 4: OK, ví dụ:", ex4_shortest_strict_sol([1, 2, 2, 3]), ex4_shortest_strict_sol([1, 2, 3, 10, 4, 2, 3, 5]))

Bài 4: OK, ví dụ: 1 4


In [31]:
# Bài 5 — LC 581
def ex5_shortest_unsorted(nums):
    n = len(nums)
    right, mx = -1, float("-inf")
    for k in range(n):
        if nums[k] < mx:
            right = k
        else:
            mx = nums[k]
    left, mn = 0, float("inf")
    for k in range(n - 1, -1, -1):
        if nums[k] > mn:
            left = k
        else:
            mn = nums[k]
    return 0 if right == -1 else right - left + 1

def ex5_brute(nums):
    s = sorted(nums)
    diff = [k for k in range(len(nums)) if nums[k] != s[k]]
    return 0 if not diff else diff[-1] - diff[0] + 1

print("Bài 5:", ex5_shortest_unsorted([2, 6, 4, 8, 10, 9, 15]), ex5_shortest_unsorted([1, 2, 3, 4]), ex5_shortest_unsorted([1]))
assert ex5_shortest_unsorted([2, 6, 4, 8, 10, 9, 15]) == 5
rng = random.Random(5)
for _ in range(1500):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(1, 12))]
    assert ex5_shortest_unsorted(nums) == ex5_brute(nums), nums
print("Bài 5: OK")

Bài 5: 5 0 0
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class Solution:
    def findLengthOfShortestSubarray(self, arr):
        n = len(arr)
        i = 0
        while i + 1 < n and arr[i] <= arr[i + 1]:   # tiền tố không giảm
            i += 1
        if i == n - 1:
            return 0
        j = n - 1
        while arr[j - 1] <= arr[j]:                  # hậu tố không giảm
            j -= 1
        ans = min(n - 1 - i, j)                      # giữ một phía
        r = j
        for l in range(i + 1):                       # ghép hai phía
            while r < n and arr[r] < arr[l]:
                r += 1
            if r == n:
                break
            ans = min(ans, r - l - 1)
        return ans
```

**Template "xoá một đoạn":** phần còn lại = tiền tố + hậu tố → tìm tiền tố/hậu tố tốt nhất → xử lý chỗ nối bằng hai con trỏ/nhị phân → đừng quên phương án giữ một phía.

**Checklist trước khi submit:**
- [ ] So sánh dùng `<=` (không giảm), chỗ nối `arr[r] >= arr[l]`.
- [ ] Return `0` ngay khi mảng đã sắp.
- [ ] Khởi tạo `ans = min(n - 1 - i, j)`.
- [ ] Con trỏ `r` bắt đầu từ `j` và chỉ tiến; dừng khi `r == n`.
- [ ] Nếu dùng nhị phân: `bisect_left(arr, arr[l], j)`.
- [ ] Thử nhanh: ví dụ → `3, 4, 0`; `[10,1,2,3]→1`; `[1,2,3,0]→1`; `[1,1,1]→0`.